# Function Calling Agent 动手实践
## 2026-05-06 | 第1周周三 | 零框架纯手写

> **学习目标**: 理解 LLM Function Calling 的底层原理，手写一个完整的 Agent 循环

**你将学到：**
- Tool Schema 的设计原则（JSON Schema 标准）
- Agent 循环的工程实现（ReAct 模式）
- DeepSeek API 的 Tool Calling 用法
- 工具安全执行（受限 eval 沙箱）

**工具清单：**
1. `get_weather` — 查天气（温度、天气状况、湿度、风速）
2. `calculate` — 数学计算（四则运算、三角函数、幂运算等）

---


## 新手导读：先把 Function Calling 想成「模型开出的工单」

这一节最容易卡住的点，是把 LLM 和工具的职责混在一起。可以先用一个很朴素的心智模型：

- LLM 不直接查天气、不直接算数；它只判断“我需要调用哪个工具”以及“参数应该是什么”。
- Tool Schema 是工具说明书，告诉模型工具名、用途、参数类型和必填字段。
- Tool Executor 才是真正干活的 Python 函数。
- Agent Loop 负责把模型的 `tool_calls` 变成真实函数调用，再把工具结果塞回对话历史。

阅读顺序建议：

1. 先看 `WEATHER_TOOL` / `CALCULATOR_TOOL`，理解 schema 是“合同”，不是执行逻辑。
2. 再看 `get_weather` / `calculate`，理解工具实现可以很普通，甚至完全不懂 LLM。
3. 最后看 `run_agent`，关注 `messages.append(...)`：Agent 的“记忆”其实就是消息历史。

常见误区：

- 不要把 Function Calling 理解成“模型会自动执行 Python”。模型只是返回结构化 JSON，执行权在你的代码里。
- `tool_choice="auto"` 表示让模型自己决定是否调用工具；不是每一轮都必须调用。
- 工具返回值最好结构化、短而清楚，否则会污染下一轮模型上下文。


## 1. 环境准备

初始化 DeepSeek 客户端。DeepSeek API 完全兼容 OpenAI SDK——只需修改 `base_url` 和 `api_key`。

> 参考文档: https://api-docs.deepseek.com/zh-cn/

In [ ]:
import json, math, os, sys
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Windows 终端 GBK 编码兼容（Jupyter 中 sys.stdout 是 OutStream，无 reconfigure 方法）
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

# DeepSeek API 完全兼容 OpenAI SDK，只需把 base_url 改成 DeepSeek 的地址
# 这意味着你可以无缝把现有 OpenAI 项目迁移到 DeepSeek，只换两行配置
client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="https://api.deepseek.com",
)

# 验证连接——list_models 是最轻量的存活探测方式
models = client.models.list()
for m in models.data:
    print(f"  {m.id}")
print("连接成功! 使用模型: deepseek-v4-flash")

## 2. Tool Schema 设计（核心中的核心）

### 什么是 Tool Schema？

Tool Schema 是 LLM 与工具之间的**"合同"**。它的本质是 JSON Schema 格式的函数签名，告诉 LLM：
- 这个工具叫什么（`name`）
- 它能干什么（`description`）
- 需要什么参数、什么类型、哪些必填（`parameters`）

### Function Calling 底层流程（八股题 10）

```
① 开发者定义 Tool Schema 列表
         ↓
② 将 tools[] 随请求发给 LLM
         ↓
③ LLM 根据用户意图二选一：
   ├── 直接回复文本  → response.choices[0].message.content
   └── 返回 tool_calls → response.choices[0].message.tool_calls[]
         ↓
④ 开发者在本地执行工具，把结果追加到对话历史
         ↓
⑤ LLM 综合结果，生成最终回复
```

### Tool Schema 设计三原则（八股题 11）

| 原则 | 说明 | 好例子 | 坏例子 |
|------|------|--------|--------|
| **description 要详细** | LLM 靠它判断何时该用 | "查询指定城市的实时天气，返回温度、湿度、风速、天气状况" | "获取天气" |
| **parameters 严格 Schema** | `required` 数组标明必填项 | `"required": ["city"]` | 所有参数都不标 required |
| **命名语义化** | name 动词+名词 | `get_weather`, `calculate` | `func1`, `do_stuff` |


### 代码：定义两个 Tool Schema

观察下面两个工具的设计差异：
- `WEATHER_TOOL` 有多个参数（city 必填 + unit 可选），用了 `enum` 约束
- `CALCULATOR_TOOL` 只有一个参数，简洁清晰

In [ ]:
# ── 工具 1: 天气查询 ──
# 注意 "type": "function" 是固定外层包装，LLM 看到这个格式才知道这是可调用的函数
WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_weather",
        # description 是 LLM 判断"该不该用这个工具"的唯一依据，写得越清楚越好
        "description": (
            "查询指定城市的实时天气信息。"
            "返回数据包含：温度、天气状况、湿度、风速。"
            "适用场景：用户询问某地天气怎么样、是否需要带伞等。"
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "城市名称，支持中文或英文，例如：北京、Tokyo、London",
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],   # enum 限制合法值，防止模型传入奇怪字符串
                    "description": "温度单位，默认celsius（摄氏度）",
                },
            },
            "required": ["city"],   # unit 不在 required 里 = 可选参数；模型会自行决定是否传
        },
    },
}

# ── 工具 2: 数学计算 ──
CALCULATOR_TOOL = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": (
            "执行数学表达式计算。"
            "支持：+-*/四则运算、**幂运算、三角函数(sin/cos/tan)、"
            "sqrt平方根、log/log10对数、abs绝对值。"
            "当用户需要精确数值计算时必须调用此工具，禁止心算。"
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "数学表达式字符串，如 '(2+3)*4'、'sqrt(144)'",
                }
            },
            "required": ["expression"],
        },
    },
}

# TOOLS 列表会整体传给 LLM，LLM 根据用户意图从中选择要调用哪个
TOOLS = [WEATHER_TOOL, CALCULATOR_TOOL]
print("Tool Schema 定义完成，共", len(TOOLS), "个工具")

# 你可以查看 JSON Schema 的完整结构
print("\nWEATHER_TOOL 的完整 JSON Schema:")
print(json.dumps(WEATHER_TOOL, indent=2, ensure_ascii=False))

## 3. 工具实现（真正干活的代码）

> **关键认知**: LLM **不执行**代码！它只输出"我想调用 `get_weather(city='北京')`"。
> 真正的函数调用、HTTP 请求、计算，全在开发者代码中完成。

### 安全注意事项

`calculate` 使用 `eval` 执行表达式，必须做沙箱隔离：
- `__builtins__` 设为空字典 → 彻底阻断 `os.system`、`__import__`、`open` 等危险调用
- 只暴露 `math` 模块的公开函数 → 白名单机制

> 生产环境建议用 `numexpr` 库替代 `eval`: https://github.com/pydata/numexpr


In [ ]:
# ── 天气查询实现（当前为模拟数据）──
# 生产环境可替换为:
#   wttr.in (免费):  requests.get(f"https://wttr.in/{city}?format=j1")
#   OpenWeatherMap:  https://openweathermap.org/api
# 函数签名和返回值格式不用改，只需换内部实现——这就是 schema 和实现分离的好处

def get_weather(city: str, unit: str = "celsius") -> dict:
    """查询城市天气（当前为模拟数据）"""
    weather_db = {
        "北京":     {"temp_c": 22, "condition": "晴",     "humidity": 40, "wind": "北风 3级"},
        "上海":     {"temp_c": 25, "condition": "多云",   "humidity": 68, "wind": "东南风 2级"},
        "广州":     {"temp_c": 29, "condition": "雷阵雨", "humidity": 85, "wind": "南风 4级"},
        "深圳":     {"temp_c": 28, "condition": "阴",     "humidity": 78, "wind": "东风 3级"},
        "杭州":     {"temp_c": 24, "condition": "小雨",   "humidity": 72, "wind": "东北风 2级"},
        "成都":     {"temp_c": 21, "condition": "阴",     "humidity": 75, "wind": "无持续风向 1级"},
        "tokyo":    {"temp_c": 18, "condition": "晴",     "humidity": 50, "wind": "北风 2级"},
        "london":   {"temp_c": 13, "condition": "小雨",   "humidity": 80, "wind": "西风 5级"},
        "new york": {"temp_c": 16, "condition": "多云",   "humidity": 55, "wind": "西南风 4级"},
        "sydney":   {"temp_c": 20, "condition": "晴",     "humidity": 45, "wind": "东风 3级"},
    }
    key = city.strip().lower()
    data = weather_db.get(key, {"temp_c": 20, "condition": "暂无数据", "humidity": 60, "wind": "未知"})
    temp = data["temp_c"]
    unit_label = "°C"
    if unit == "fahrenheit":
        temp = round(temp * 9/5 + 32, 1)
        unit_label = "°F"
    return {
        "city": city, "temperature": temp, "unit": unit_label,
        "condition": data["condition"], "humidity": f"{data['humidity']}%", "wind": data["wind"]
    }

# ── 计算器实现（安全沙箱）──
def calculate(expression: str) -> dict:
    """
    安全执行数学表达式。
    __builtins__ 设为空字典：彻底阻断 os.system、__import__、open 等危险调用。
    只暴露 math 模块的函数：白名单机制，这是面试里常被问到的安全设计点。
    """
    allowed = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
    allowed.update({"abs": abs, "round": round, "min": min, "max": max, "pow": pow})
    try:
        result = eval(expression, {"__builtins__": {}}, allowed)
        return {"expression": expression, "result": result, "error": None}
    except Exception as e:
        return {"expression": expression, "result": None, "error": str(e)}

# ── 工具路由表 ──
# 字典映射是新增工具的扩展点：只需加一行，不用改调度逻辑
TOOL_EXECUTORS = {"get_weather": get_weather, "calculate": calculate}

def execute_tool(name: str, args: dict) -> str:
    """工具统一调度入口，返回 JSON 字符串（LLM 会把这段字符串当成工具返回值读取）"""
    func = TOOL_EXECUTORS.get(name)
    if func is None:
        return json.dumps({"error": f"未知工具: {name}"}, ensure_ascii=False)
    try:
        return json.dumps(func(**args), ensure_ascii=False)
    except Exception as e:
        return json.dumps({"error": str(e)}, ensure_ascii=False)

# 快速验证——确保工具能正常工作
print(">>> get_weather('北京'):")
print("   ", execute_tool("get_weather", {"city": "北京"}))

print("\n>>> calculate('2**10 + sqrt(256)'):")
print("   ", execute_tool("calculate", {"expression": "2**10 + sqrt(256)"}))
print("\n工具就绪!")

## 4. 系统提示词（System Prompt）

System Prompt 定义 Agent 的角色边界和行为规范。它会被放在每条消息历史的最前面。

**设计要点：**
- 列举可用工具及其用途
- 明确何时调用工具、何时不调
- 规定回复风格（语言、格式、简洁度）
- 鼓励并行调用（一次调多个工具，减少轮次）


In [ ]:
SYSTEM_PROMPT = """你是一个具备工具调用能力的智能助手。你拥有以下工具：

1. get_weather — 查询任意城市的实时天气（温度、天气状况、湿度、风速）
2. calculate   — 执行数学表达式计算（支持四则运算、幂运算、三角函数等）

行为准则：
- 用户询问天气相关信息时，主动调用 get_weather
- 用户需要数值计算时，调用 calculate，禁止自行心算
- 收到工具返回结果后，用流畅的中文向用户转述
- 如果用户同时问了天气和计算，可以一次调用多个工具（并行调用）
- 保持回答简洁、信息密度高"""

# System Prompt 放进每条对话历史的最前面，等效于给 LLM 设定工作职责书
print("System Prompt 已加载")
print("长度:", len(SYSTEM_PROMPT), "字符")

## 5. Agent 循环 —— 核心中的核心

Agent 循环就是 **ReAct (Reasoning + Acting)** 模式的工程实现：

```
用户输入
  │
  ▼
┌──────────┐     ┌──────────────┐     ┌───────────┐
│   LLM    │────→│  tool_calls? │──是──→│  执行工具   │──┐
└──────────┘     └──────┬───────┘     └───────────┘  │
       ↑                │ 否                          │
       │                ▼                             │
       │         ┌───────────┐                        │
       │         │  返回文本   │                        │
       │         │  (结束)    │                        │
       │         └───────────┘                        │
       │                                              │
       └──────────────── 追加结果到历史 ────────────────┘
```

**关键设计细节：**
- `tool_choice="auto"` — LLM 自行判断是否调工具
- `temperature=0.0` — 工具调用需要确定性，不用随机性
- `tool_call_id` 必须对上——LLM 靠它关联"哪个调用对应哪个结果"
- `max_turns=10` — 安全上限，防止死循环
- `messages.append(msg.model_dump())` — 将含 tool_calls 的 assistant 消息原样保存


### 代码：Agent 主循环

下面是 `run_agent` 函数——整个 Agent 系统的核心。**建议打断点逐行调试**，观察消息历史的变化。

In [ ]:
def run_agent(
    client: OpenAI,
    user_query: str,
    model: str = "deepseek-v4-flash",
    max_turns: int = 10,
    verbose: bool = True,
) -> str:
    """
    Agent 主循环 —— ReAct (Reasoning + Acting) 模式的核心实现。

    参数:
        client:     DeepSeek 客户端
        user_query: 用户输入的自然语言问题
        model:      模型 ID (deepseek-v4-flash / deepseek-v4-pro)
        max_turns:  最大 LLM 交互轮次（安全上限，防止死循环）
        verbose:    是否打印每轮的调用详情

    返回:
        Agent 的最终文本回复
    """
    # 消息历史是 Agent 的"记忆"——每一轮 LLM 都能看到完整的对话上下文
    # 如果你不 append 工具结果，下一轮 LLM 会不知道自己调用了什么
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for turn in range(1, max_turns + 1):
        # ── 第①步：调用 LLM ──
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",     # "auto"=让模型自己决定是否调工具；"none"=强制不调；"required"=强制调
            temperature=0.0,        # 工具调用需要确定性结果，设 0 消除随机性
        )

        msg = response.choices[0].message

        # ── 第②步：判断 LLM 意图 ──
        if msg.tool_calls:
            # ====== 分支 A: LLM 要求调用工具 ======
            if verbose:
                names = [tc.function.name for tc in msg.tool_calls]
                print(f"\n[轮次 {turn}] 调用工具: {', '.join(names)}")

            # 关键！必须把含有 tool_calls 的 assistant 消息原样 append 到历史
            # 跳过这步会导致 API 报错：缺少对应 tool_call_id 的 tool 消息
            messages.append(msg.model_dump())

            for tc in msg.tool_calls:
                tool_name = tc.function.name
                tool_args = json.loads(tc.function.arguments)

                if verbose:
                    print(f"  IN  {tool_name}({json.dumps(tool_args, ensure_ascii=False)})")

                # 本地执行工具（调用真正干活的 Python 函数）
                result = execute_tool(tool_name, tool_args)

                if verbose:
                    print(f"  OUT {result}")

                # tool_call_id 必须和 assistant 消息里的 id 对应
                # LLM 靠这个 id 把"我要调用X"和"X的返回结果"关联起来
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
            # 本轮结束，回到循环开头——LLM 看到工具结果后再决定下一步

        else:
            # ====== 分支 B: LLM 直接文本回复 = 最终答案 ======
            if verbose:
                print(f"\n[轮次 {turn}] 最终回复 (文本)")
            return msg.content

    # 走到这里说明 max_turns 内没有得到文字回复，一般是模型循环调工具导致的
    return "处理超时，请将问题拆分为更小的子问题。"

print("Agent 循环函数已定义，可以开始交互测试!")

## 6. 交互测试

### 6.1 单次查询测试

修改下面 `query` 变量的值，然后运行这个 Cell。尝试不同的问法，观察 Agent 的调用链路。

**试试这些：**
- `"深圳今天天气怎么样？"`
- `"帮算一下 12345 * 67890"`
- `"计算 sin(pi/6) + cos(pi/3)"`


In [ ]:
# ===== 修改这行，测试不同的查询 =====
query = "北京今天天气怎么样？适合出去玩吗？"
# =====================================

print(f"用户: {query}")
answer = run_agent(client, query, verbose=True)
print(f"\nAgent 最终回复:\n{answer}")

### 6.2 更多测试场景

以下是一些值得尝试的查询——覆盖单工具、并行调用、多步推理等场景。逐个运行观察 LLM 是如何"思考"的。

In [ ]:
# 场景 A: 单工具 — 复杂两步计算
# 注意 LLM 会分两轮调用 calculate，而不是一次给出两步
query = "帮我算一下 2 的 20 次方是多少？然后再把结果除以 1024"
print(f"用户: {query}")
answer = run_agent(client, query, verbose=True)
print(f"\nAgent:\n{answer}")

In [ ]:
# 场景 B: 并行调用 — 一次查两个城市
# 当 LLM 判断两个调用互相独立时，会在同一个 tool_calls 列表里返回两个调用
# 这样只需一轮 API 调用，比顺序调用省了一次往返
query = "上海和广州现在的天气分别怎么样？哪个更适合出门？"
print(f"用户: {query}")
answer = run_agent(client, query, verbose=True)
print(f"\nAgent:\n{answer}")

In [ ]:
# 场景 C: 多步推理 — 先查天气，再根据结果计算
# 这里 calculate 依赖 get_weather 的返回值，所以 LLM 必须分两轮执行（无法并行）
query = "北京现在多少度？如果北京比成都热 5 度，成都应该是多少度？"
print(f"用户: {query}")
answer = run_agent(client, query, verbose=True)
print(f"\nAgent:\n{answer}")

In [ ]:
# 场景 D: 英文输入 — 测试跨语言处理
# LLM 会用英文城市名调用工具，工具内部做 .lower() 处理以匹配数据库键
query = "What's the weather in Tokyo and New York? Answer in Chinese please."
print(f"用户: {query}")
answer = run_agent(client, query, verbose=True)
print(f"\nAgent:\n{answer}")

In [ ]:
# 场景 E: 几何计算 — 三角函数
# math 模块的 sin/cos 接受弧度；LLM 知道要先把角度转成弧度（乘以 pi/180）
query = "一个角度为30度的直角三角形，斜边长为10，请帮我计算两条直角边的长度"
print(f"用户: {query}")
answer = run_agent(client, query, verbose=True)
print(f"\nAgent:\n{answer}")

### 6.3 自由输入

输入你自己的问题，看看 Agent 如何应对。

**挑战题目：**
- 问一个和天气、计算都无关的问题 → Agent 会乱调工具吗？
- 问"帮我算 3+5，顺便查一下北京天气" → 会并行调用吗？
- 用拼音或拼错的英文城市名 → 能正确处理吗？


In [ ]:
# ===== 自由发挥区：输入任何你想问的 =====
query = input("请输入你的问题: ").strip()
if not query:
    query = "今天深圳天气如何？帮我算一下 1024 * 768 的结果"
    print(f"(使用默认问题): {query}")

print()
answer = run_agent(client, query, verbose=True)
print(f"\nAgent 最终回复:\n{answer}")

## 7. 深入理解：观察消息历史

想要真正理解 Agent 循环，必须看**消息历史**是怎么在每轮变化。

下面这个版本额外返回完整的 `messages` 列表，你可以看到：
- 每一轮 LLM 收到了什么
- tool_calls 是怎么 append 的
- tool 结果是怎么 append 的


In [ ]:
def run_agent_with_history(
    client: OpenAI,
    user_query: str,
    model: str = "deepseek-v4-flash",
    max_turns: int = 10,
) -> tuple[str, list]:
    """与 run_agent 功能相同，但额外返回完整的 messages 历史，方便调试观察"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS,
            tool_choice="auto", temperature=0.0,
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append(msg.model_dump())
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                result = execute_tool(tc.function.name, args)
                messages.append({
                    "role": "tool", "tool_call_id": tc.id, "content": result,
                })
        else:
            return msg.content, messages

    return "超时", messages


# 执行并查看消息历史
final_answer, history = run_agent_with_history(client, "北京天气怎么样？")

print("=" * 60)
print("完整消息历史 (共", len(history), "条)")
print("=" * 60)

for i, msg in enumerate(history):
    role = msg["role"]
    content = msg.get("content", "")

    if role == "tool":
        # tool 消息：tool_call_id 是关联键，把工具返回和对应调用绑定
        print(f"\n[{i}] role=tool  tool_call_id={msg['tool_call_id'][:8]}...")
        print(f"    content: {content[:300]}")
    elif role == "assistant" and "tool_calls" in msg:
        # assistant 消息里有 tool_calls：说明这一轮 LLM 要求调工具
        for tc in msg["tool_calls"]:
            print(f"\n[{i}] role=assistant -> tool_call: {tc['function']['name']}")
            print(f"    arguments: {tc['function']['arguments']}")
    elif role == "assistant":
        # assistant 消息里没有 tool_calls：这是最终文本回复
        print(f"\n[{i}] role=assistant (final)")
        print(f"    content: {content[:300]}")
    else:
        # system 或 user
        preview = (content or "")[:200].replace("\n", " ")
        print(f"\n[{i}] role={role}: {preview}...")

print(f"\n\n最终回复:\n{final_answer}")

## 8. 参考资料

| 主题 | 链接 | 说明 |
|------|------|------|
| OpenAI Function Calling | https://platform.openai.com/docs/guides/function-calling | 官方文档，最权威 |
| Anthropic Tool Use | https://docs.anthropic.com/en/docs/build-with-claude/tool-use | Claude 的工具使用指南 |
| Anthropic Writing Effective Tools | https://www.anthropic.com/engineering/writing-tools-for-agents | 工具设计最佳实践 |
| DeepSeek API | https://api-docs.deepseek.com/zh-cn/ | DeepSeek API 文档 |
| JSON Schema | https://json-schema.org/ | Tool Schema 的标准格式 |
| Building Effective Agents | https://www.anthropic.com/research/building-effective-agents | Anthropic Agent 设计哲学 |
| ReAct 论文 | https://arxiv.org/abs/2210.03629 | Reasoning + Acting 原始论文 |

### 明日预告（2026-05-07）

把今天的 Agent 改造成 **MCP Server**！

参考: https://modelcontextprotocol.io

MCP (Model Context Protocol) 是标准化的工具暴露协议——把今天的硬编码工具调用变成标准的 Client-Server 架构，让任何 MCP 客户端都能调用你的工具。

## 9. 核心知识点回顾

### 今天你掌握了什么

| # | 知识点 | 对应位置 |
|---|--------|---------|
| 1 | **Tool Schema 设计** — JSON Schema 格式，name + description + parameters + required | Cell 2-3 |
| 2 | **Function Calling 底层** — LLM 不执行代码，只输出调用意图；真正执行在开发者侧 | Cell 4 |
| 3 | **Agent 循环** — ReAct 模式的实现，消息历史的追加与管理 | Cell 5-7 |
| 4 | **安全设计** — 受限 eval 沙箱，`__builtins__` 置空 | Cell 4 |
| 5 | **DeepSeek API** — 完全兼容 OpenAI SDK，只需改 `base_url` | Cell 1 |

### 对应八股题

| 题号 | 主题 | 对应 Cell |
|------|------|-----------|
| 题 10 | Function Calling 底层原理 | Cell 2 (理论) + Cell 7 (实现) |
| 题 11 | Tool Schema 设计三原则 | Cell 2 (原则) + Cell 3 (代码) |

### 代码文件

- `exercises/w1d1-function-calling/agent.py` — 完整可运行脚本
- `function_calling_agent.ipynb` — 交互式学习 Notebook（当前文件）

---

> Agent 循环 + Tool Schema 是整个 agent 系统最底层的核心能力。
> 后面的 MCP、LangGraph、多 agent 协作都是在此基础上的扩展。
> 建议把 `run_agent` 函数的每行代码都理解透——它就是所有 agent 框架的"最小公分母"。

## 学习检查清单

读完这节后，建议你能回答：

- Tool Schema 里的 `description` 为什么要写清楚？
- `required` 字段少写或多写会带来什么问题？
- Agent 为什么要把工具结果以 `role="tool"` 追加回消息历史？
- `calculate` 为什么要限制 `eval` 的 `__builtins__`？
- 如果模型连续调用工具超过 `max_turns`，应该如何防止死循环？
